# Python 三大进阶特性：迭代器 / 生成器 / 装饰器

---

**这份笔记解决什么问题？**

写 Python 时你一定见过这三个词，但它们看起来"很玄"。其实它们解决的是三个非常具体的问题：

| 特性 | 解决什么问题 | 一句话理解 |
|---|---|---|
| 迭代器 Iterator | 怎么让 `for` 能遍历我自己的东西 | 一个会"叫号"的排队机 |
| 生成器 Generator | 数据太多、不能一次性全塞进内存 | 水龙头，要多少接多少 |
| 装饰器 Decorator | 想给一堆函数加同样的"前后处理" | 给函数套一个手机壳 |

**学习路径建议**：先看懂迭代器，生成器自然就懂了；装饰器相对独立，可以最后看。


---
## 一、可迭代对象与迭代器

### 1.1 先问一个问题：`for` 到底是怎么工作的？

```python
for fruit in ['apple', 'banana', 'cherry']:
    print(fruit)
```

这行代码背后，Python 其实做了三件事：

1. 对列表调用 `iter()`，拿到一个**迭代器**对象
2. 反复调用这个迭代器的 `__next__()`，一个个取出元素
3. 取完了，迭代器会抛出一个 `StopIteration` 异常，`for` 捕获到它就安静地结束循环

所以 `for` 不是什么魔法，它只是把这个"取一次、再取一次"的过程包装起来了。

### 1.2 类比：排队叫号机

迭代器就像银行门口的**叫号机**：

- 它手里有一份名单（数据）
- 每按一次按钮（`next()`），就吐出下一个号码
- 叫完了再说"没有了"（`StopIteration`）
- **重点：它只往前走，不回头。** 叫过的号再按也不会重复出现

这个"一次性"的特性是理解迭代器的关键，后面很多"坑"都和它有关。


### 1.3 手动体验一遍 `iter()` 和 `next()`


In [ ]:
from typing import Iterable

fruits = ['apple', 'banana', 'cherry']

# 判断一个东西能不能被 for 遍历：它是不是 Iterable（可迭代对象）
print('列表是可迭代对象吗？', isinstance(fruits, Iterable))

# iter() 把可迭代对象变成迭代器
fruits_iterator = iter(fruits)
print('迭代器对象：', fruits_iterator)

# next() 每调用一次，取出一个元素
print(next(fruits_iterator))   # apple
print(next(fruits_iterator))   # banana
print(next(fruits_iterator))   # cherry

# 取完了再取，就会抛 StopIteration —— for 循环内部就是靠捕获它来结束的
try:
    print(next(fruits_iterator))
except StopIteration:
    print('已经取完了，抛出 StopIteration')


**注意上面输出的 `fruits_iterator` 长这样**：`<list_iterator object at 0x...>`

它就是个"状态机"，内部记着"我读到第几个了"。所以：

- 迭代器**可以被 `for` 遍历**，但遍历完之后就"空了"
- 想再遍历一遍，必须重新 `iter()` 一次


In [ ]:
it = iter(['a', 'b'])
print('第一次遍历：', list(it))
print('第二次遍历：', list(it))   # 空的！因为迭代器已经走到头了

it2 = iter(['a', 'b'])
print('重新 iter 后：', list(it2))


### 1.4 自己写一个迭代器

只要一个类同时实现了这两个方法，它就是迭代器：

- `__iter__()`：返回迭代器自己（所以迭代器一定可迭代）
- `__next__()`：返回下一个值；没值了要 `raise StopIteration`

下面这个例子实现的是"从 1 数到 n"的迭代器：


In [ ]:
class MyIterator:
    """从 1 数到 n 的迭代器"""

    def __init__(self, n):
        self.n = n          # 数到几为止
        self.index = 1      # 当前数到几了（这就是"状态"）

    def __iter__(self):
        # 迭代器要求 __iter__ 返回自己
        return self

    def __next__(self):
        if self.index > self.n:
            # 关键：取完了必须抛这个异常，for 才知道该停了
            raise StopIteration
        value = self.index
        self.index += 1     # 每取一次，状态往前走一格
        return value


print('用 for 遍历自定义迭代器：', list(MyIterator(5)))
print('用 next() 手动取：', next(MyIterator(3)))


**`enumerate` 其实也是迭代器**，它每次吐出一个 `(下标, 元素)` 元组：


In [ ]:
for index, fruit in enumerate(fruits):
    print(index, fruit)

# start 参数可以改变起始编号，写报表时很实用（从 1 开始数第几名）
for rank, fruit in enumerate(fruits, start=1):
    print(f'第 {rank} 名：{fruit}')


> **原始笔记的修正说明**
>
> 原笔记里的 `MyIterator` 是"直接把一个列表拿过来逐个吐"，逻辑没问题，但少了"为什么要有 index"这个最关键的铺垫。这里改成了"从 1 数到 n"，因为计数器是迭代器**唯一的状态**，这样更容易理解"迭代器 = 状态机"这件事。


---
## 二、生成器：自动帮你写好的迭代器

### 2.1 为什么需要生成器？

假设你要处理 1000 万个数字。

- **列表方案**：`[i * i for i in range(10_000_000)]` —— 一次性把这 1000 万个数字全算出来存进内存，可能直接爆内存
- **生成器方案**：`(i * i for i in range(10_000_000))` —— 一个都不算，等你 `next()` 的时候才算那一个

### 2.2 类比：倒一桶水 vs 开水龙头

| | 列表 | 生成器 |
|---|---|---|
| 像什么 | 一桶水，先盛满再用 | 水龙头，拧开就流 |
| 内存占用 | 装下所有数据 | 只记住"当前状态" |
| 能不能回头 | 能，随时 `[i]` 取任意一个 | 不能，只能往前 |
| 适合场景 | 数据不大、要反复读 | 数据量大、只读一遍 |

### 2.3 `yield`：生成器的灵魂

普通函数用 `return`，**一返回函数就结束了**。
生成器函数用 `yield`，**它会把值交出去，然后"暂停"在那里**，下次再调用继续往下走。

```python
def my_generator(n):
    for i in range(n):
        yield i * i     # 交出一个值，然后暂停
```

只要函数体里出现了 `yield`，这个函数就**不再是一个普通函数**了 ——
调用它不会执行任何代码，而是返回一个生成器对象。


In [ ]:
def my_generator(n):
    print('  [生成器被启动了]')
    for i in range(n):
        print(f'  [准备产出第 {i} 个]')
        yield i * i


gen = my_generator(3)
print('调用函数后，什么都没打印 —— 因为里面的代码一行都没执行')
print('它返回的是：', type(gen))
print('现在开始 next：')
print(next(gen))    # 这时才真正开始跑函数体
print(next(gen))
print(next(gen))


**看到关键了吗？** 函数体是在你第一次 `next()` 时才开始执行的，而且每 `next()` 一次只跑到下一个 `yield` 就停住。这就是"惰性求值"。


### 2.4 生成器表达式：`()` 版列表推导式

把列表推导式的方括号换成圆括号，就得到生成器表达式：


In [ ]:
temp = (i for i in range(10))
print('类型：', type(temp))
print('内容：', temp)          # 看不到内容！生成器不存数据

# 但它可以被 for 遍历，也可以被 next 逐个取出
print('前 5 个：', [next(temp) for _ in range(5)])

# 内存占用的直观对比：下面这行如果是列表，会真的造出 100 万个整数
big_gen = (i for i in range(1_000_000))
import sys
print('生成器对象占用的字节数：', sys.getsizeof(big_gen))
big_list = [i for i in range(1_000_000)]
print('列表占用的字节数：', sys.getsizeof(big_list), '（相差悬殊）')


### 2.5 实战：用生成器写斐波那契数列

> **原始笔记修正说明**
>
> 原笔记里这样写：
>
> ```python
> def fibonacci(n):
>     a, b = 0, 1
>     for _ in range(n):
>         a, b = b, a + b
>         print(a)
>     yield a
>
> fibonacci(10)      # 这里什么都不会输出！
> ```
>
> **问题在哪？** 函数体里有 `yield`，所以 `fibonacci` 是生成器函数。
> `fibonacci(10)` 只是**创建了一个生成器对象**，函数体一行都没执行，所以不会有任何输出。
> 而且 `yield` 写在了循环**外面**，意味着只会产出一个值，也不符合"数列"的语义。
>
> 下面是修正版：


In [ ]:
def fibonacci(n):
    """生成前 n 个斐波那契数"""
    a, b = 0, 1
    for _ in range(n):
        yield a             # yield 写在循环【里面】，才能产出 n 个值
        a, b = b, a + b     # 元组解包，等价于同时更新 a 和 b


# 正确用法：用 for 遍历它
print('前 10 个斐波那契数：')
for x in fibonacci(10):
    print(x, end=' ')
print()

# 也可以按需取前几个（这才是生成器真正的好处：不需要的不算）
fib = fibonacci(1000000)            # 一千万个也不占内存
print('取前 8 个：', [next(fib) for _ in range(8)])


### 2.6 生成器的两个实用配套函数

- `yield from`：把另一个可迭代对象的值"转发"出来，省掉一层循环
- 生成器只能遍历一次，如果想留着反复用，记得先 `list()` 存下来


In [ ]:
def chain_demo():
    yield from [1, 2, 3]
    yield from ['a', 'b']

print(list(chain_demo()))


---
## 三、装饰器：给函数套个"壳"

### 3.1 先理解一个前提：函数也是对象

在 Python 里，函数和数字、字符串一样，**是可以被传来传去的值**。
你可以把它赋给变量、当参数传、当返回值返回。


In [ ]:
def greet():
    return '你好'


f = greet              # 注意：没有括号！这是把函数本身赋给 f
print('f 是什么：', f)
print('调用 f() 的结果：', f())


### 3.2 类比：装饰器就是"手机壳"

你的手机（原函数）功能不变，但套上壳以后：

- 多了一层保护（加了日志、异常处理）
- 多了一些功能（加了计时、权限校验）
- **但手机本身没有被切开改造**

装饰器就是**不修改原函数代码**，给函数**动态加上额外功能**的做法。

### 3.3 手写第一个装饰器

关键点：装饰器本身是一个**接收函数、返回函数**的函数。


In [ ]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        # *args / **kwargs 是为了把原函数收到的任意参数原样转发过去
        print('  [函数执行前]')
        result = func(*args, **kwargs)
        print('  [函数执行后]')
        return result
    return wrapper


@my_decorator                    # 这一行等价于：say_hello = my_decorator(say_hello)
def say_hello():
    print('  Hello!')


# > 原始笔记修正说明：原笔记定义了装饰器和 say_hello，但没有【调用】它，
# > 所以运行后看不到任何效果。这里补上调用。
say_hello()


### 3.4 装饰器最实用的场景：计时


In [ ]:
import time

def timer(func):
    """打印函数运行耗时的装饰器"""
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        cost = time.perf_counter() - start
        print(f'  {func.__name__} 耗时 {cost:.4f} 秒')
        return result
    return wrapper


@timer
def slow_sum(n):
    total = 0
    for i in range(n):
        total += i
    return total


print('结果：', slow_sum(2_000_000))
# 同样的计时需求，换成另一个函数只需要加一行 @timer，不用改函数本身。


### 3.5 一个必须知道的坑：`functools.wraps`

上面的 `wrapper` 把原函数的"身份信息"覆盖掉了。
加个 `@wraps(func)` 就能保住原函数的名字和文档字符串：


In [ ]:
import functools

def deco_bad(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

def deco_good(func):
    @functools.wraps(func)        # 把 func 的元信息复制到 wrapper 上
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper


@deco_bad
def func_a():
    """这是 func_a 的说明文档"""

@deco_good
def func_b():
    """这是 func_b 的说明文档"""

print('不加 wraps：', func_a.__name__, '|', func_a.__doc__)
print('加了 wraps：', func_b.__name__, '|', func_b.__doc__)


看出来了吗？不加 `wraps` 的话，`func_a.__name__` 变成了 `wrapper`，
文档字符串也丢了。这在排查问题、自动生成接口文档时会带来很大麻烦，**所以装饰器一律加 `@functools.wraps`**。

### 3.6 带参数的装饰器（进阶）

想给装饰器传配置项（比如"重试 3 次"），就要再套一层函数：


In [ ]:
def repeat(times):
    """让被装饰的函数重复执行 times 次"""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for i in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator


@repeat(times=3)
def cheer():
    print('  加油！')

cheer()      # 会打印三次
# 结构记法：repeat(参数) -> decorator(函数) -> wrapper(真正的调用)
# 三层对应"要参数、要函数、要执行"这三件事。


---
## 四、三者关系一张图看懂

```
可迭代对象 (Iterable)         比如 list / str / dict
       │  调用 iter()
       ▼
  迭代器 (Iterator)           有 __iter__ 和 __next__
       │
       ├── 手动实现：写一个类，实现两个魔法方法
       └── 自动生成：写一个带 yield 的函数 —— 这就是【生成器】

装饰器 (Decorator) 是独立的一支：
       │
       └── 接收函数 → 返回增强后的新函数
```

**一句话总结**：

- 迭代器 = 一个**按顺序吐值**的对象
- 生成器 = **用 `yield` 自动写成的迭代器**，省内存
- 装饰器 = **给函数加外壳**，不改原函数代码就扩展功能


---
## 五、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 迭代器遍历一次就空了 | 第二次 `for` 什么都不输出 | 重新 `iter()`，或先 `list()` 存下来 |
| 调用生成器函数以为会执行 | 什么输出都没有 | 它有 `yield`，只是返回生成器对象，要遍历它 |
| `yield` 写在了循环外 | 只产出 1 个值 | 把 `yield` 挪到循环里面 |
| 忘了 `raise StopIteration` | 死循环 / 取值不停 | 手写迭代器时越界必须抛这个异常 |
| 装饰器忘了 `@functools.wraps` | 函数名变成了 `wrapper` | 一律加上 `@wraps(func)` |
| 忘了调用被装饰的函数 | 装饰器"不生效" | 装饰器是包装，不是自动执行 |
| `return` 和 `yield` 混用 | 生成器提前结束 | 生成器里用 `yield`，别用 `return` 返回值 |


---
## 六、动手练习

**练习 1（迭代器）**：写一个 `Countdown(n)` 迭代器，从 n 倒数到 1。

**练习 2（生成器）**：写一个生成器 `read_lines(path)`，逐行读取一个大文件。
思考：为什么这比 `f.readlines()` 好？（提示：`readlines()` 会把整个文件读进内存）

**练习 3（生成器）**：用生成器表达式，求出 1~1000 里所有能被 3 整除的数的和。

**练习 4（装饰器）**：写一个 `@log` 装饰器，每次调用函数时打印
`调用 xxx，参数是 (...) ，返回值是 ...`。

**练习 5（综合）**：
写一个装饰器把函数的返回值缓存起来（同一个参数第二次调用直接返回上次的结果）。
提示：用字典存"参数 -> 结果"，键可以用 `str(args)`。

---

**答案思路（先自己做）**

```python
# 练习 1
class Countdown:
    def __init__(self, n):
        self.n = n
    def __iter__(self):
        return self
    def __next__(self):
        if self.n <= 0:
            raise StopIteration
        self.n -= 1
        return self.n + 1

# 练习 2
def read_lines(path):
    with open(path, encoding='utf-8') as f:
        for line in f:          # 文件对象本身就是迭代器
            yield line.rstrip()

# 练习 3
print(sum(i for i in range(1, 1001) if i % 3 == 0))
```
